# Shift-Scan: what is left (`last.ipynb`, rewritten after re-checking Drive)

Drive check: 2026-11-09. Evidence comes from the Drive contents and the executed outputs inside
`ShiftScan_Final_Experiments_Colab_DriveFolders.ipynb`.

## What changed since the previous version
| Finding | Consequence |
|---|---|
| The previous `last.ipynb` failed at `drive.mount(..., force_remount=True)` ("credential propagation was unsuccessful") | Section 1 mounts without `force_remount`, retries, and has a manual-upload fallback |
| Files are split across two Google accounts. The account running Colab has the v2 results only as `shift_scan_v2_results-...zip` (541 MB), not as a folder | Section 2 finds the folder **or** unzips the archive automatically |
| In that account the project folder holds no `.py` files and no checkpoints (0 and 0) | Training code must come from `Shift_Scan_v2_Final_Colab.ipynb`; Section 4 helps you find it |
| The recovered 28-record JSONL **already contains WikiText PPL, LAMBADA accuracy and a long-context gap** for the 4M-token FP / abs-mean / TWN students | My earlier "no zero-shot data" note was wrong. Section 3 analyses this for free (no GPU) |
| `ShiftScan_Final_Experiments_Colab_DriveFolders` ran only setup cells: `COMMAND_TEMPLATE` empty, `RUN_EXPERIMENTS=False`, empty metrics CSV | Still **zero** matched A0-A3 runs |

## Still left (priority order)
| # | Item | Status |
|---|---|---|
| 1 | Fix environment (mount, accounts, unzip, locate checkpoint) | Section 1-2 |
| 2 | Free analysis of recovered WikiText / LAMBADA / long-context data | Section 3 |
| 3 | Wire the real training code | Section 4 |
| 4 | **Matched A0-A3 factorial, 3 seeds** (the PoT-decay claim has no evidence yet) | Section 5 |
| 5 | Power check: is 3 seeds enough to see a decay effect? | Section 6 |
| 6 | Component sensitivity (dt, B, C, decay, state) | Section 7 |
| 7 | Long-context test on the existing 4M checkpoint | Section 8 |
| 8 | Zero-shot for `v2_best` and A-cells (recovered rows may not cover them) | Section 9 |
| 9 | Kernel speed (Triton was ~82x slower than FP16) | Section 10 |
| 10 | Claim check and paper | Section 11-12 |

## 1. Setup and robust Drive mount

In [ ]:
import os, sys, json, time, math, random, glob, zipfile, shutil, itertools
from pathlib import Path
import numpy as np, pandas as pd

def mount_drive(retries=2):
    # No force_remount: that flag is what raised "credential propagation was unsuccessful".
    if Path('/content/drive/MyDrive').exists():
        print('Drive already mounted'); return True
    try:
        from google.colab import drive
    except ImportError:
        print('Not running in Colab; skipping mount'); return False
    for i in range(retries + 1):
        try:
            drive.mount('/content/drive')
            if Path('/content/drive/MyDrive').exists(): return True
        except Exception as e:
            print(f'mount attempt {i+1} failed:', repr(e)[:160])
            try: drive.flush_and_unmount()
            except Exception: pass
            time.sleep(3)
    return False

DRIVE_OK = mount_drive()
print('DRIVE_OK =', DRIVE_OK)
if not DRIVE_OK:
    print('FIX: Runtime > Disconnect and delete runtime, reopen this notebook, run this cell again, and accept the popup '
          'with the SAME Google account that owns the project files. If it still fails, upload the needed files by hand '
          '(see the next cell).')

In [ ]:
# Optional fallback if Drive will not mount: upload the small files by hand.
# Needed for the free analysis: shift_scan_runs_recovered*.jsonl
# Needed for the A0-A3 runs: best_config.json (and the checkpoint only for Sections 8-9).
#
# from google.colab import files
# up = files.upload()          # then set MANUAL_DIR = Path('/content')
MANUAL_DIR = Path('/content')

## 2. Locate project files (folder **or** zip)
Searches `MyDrive` for the v2 results folder, falls back to unzipping `shift_scan_v2_results*.zip`, and prints what it found.
If `best_config.json` is missing it uses the values recorded in your earlier audit, so the notebook still runs.

In [ ]:
MYDRIVE = Path('/content/drive/MyDrive') if DRIVE_OK else MANUAL_DIR
WORK = Path('/content/shiftscan_work'); WORK.mkdir(exist_ok=True)

def find_first(patterns, roots):
    for r in roots:
        r = Path(r)
        if not r.exists(): continue
        for p in patterns:
            hits = sorted(r.rglob(p))
            if hits: return hits[0]
    return None

v2_dir = MYDRIVE / 'shift_scan_v2_results'
if not v2_dir.exists():
    zips = sorted(MYDRIVE.glob('shift_scan_v2_results*.zip'))
    if zips:
        print('Found archive:', zips[0].name, round(zips[0].stat().st_size / 1e6), 'MB')
        print('Free disk GB:', round(shutil.disk_usage('/content').free / 1e9, 1))
        with zipfile.ZipFile(zips[0]) as z:
            names = z.namelist(); print('members:', len(names))
            for n in names[:25]: print('  ', n)
            z.extractall(WORK / 'v2_unzipped')
        v2_dir = WORK / 'v2_unzipped'
    else:
        v2_dir = MANUAL_DIR
print('v2 search root:', v2_dir)

CKPT       = find_first(['ckpt_v2_best_s0_t4000000.pt'], [v2_dir, MYDRIVE])
BEST_JSON  = find_first(['best_config.json'], [v2_dir, MYDRIVE])
RUNS_V2    = find_first(['runs.jsonl'], [v2_dir])
SENS_CSV   = find_first(['stage1_sensitivity.csv'], [v2_dir, MYDRIVE])
ABL_CSV    = find_first(['stage2_ablation.csv'], [v2_dir, MYDRIVE])
RECOVERED  = find_first(['shift_scan_runs_recovered*.jsonl'], [MYDRIVE / 'shift_scan_recovered_data', MYDRIVE, MANUAL_DIR])
V2_NB      = find_first(['Shift_Scan_v2_Final_Colab.ipynb'], [MYDRIVE])
LOG100     = find_first(['log.jsonl'], [MYDRIVE / 'shift_scan_100m_runs'])

DEFAULT_BEST = dict(qcfg=dict(ternary='twn', decay=None, ramp=0.2, learn_scale=True,
                              fp_proj=['0:out', '13:in', '23:out', '0:in', '3:in', '23:in']),
                    lr=0.001, tokens=4_000_000, model_size='130m', code='shiftscan-colab-v2-final')
BEST = json.load(open(BEST_JSON)) if BEST_JSON else DEFAULT_BEST

found = {k: (str(v) if v else None) for k, v in dict(CKPT=CKPT, BEST_JSON=BEST_JSON, RUNS_V2=RUNS_V2, SENS_CSV=SENS_CSV,
         ABL_CSV=ABL_CSV, RECOVERED=RECOVERED, V2_NB=V2_NB, LOG100=LOG100).items()}
display(pd.DataFrame(list(found.items()), columns=['item', 'path']))
if not BEST_JSON: print('best_config.json not found; using default recorded in the earlier audit.')

OUT = (MYDRIVE / 'shift_scan_last_phase') if DRIVE_OK else (WORK / 'last_phase')
OUT.mkdir(exist_ok=True)
RESULTS = OUT / 'runs_last.jsonl'   # append-only, resumable

def append_jsonl(path, row):
    with open(path, 'a') as f: f.write(json.dumps(row) + '\n')

def read_jsonl(path):
    if not path or not Path(path).exists(): return []
    out = []
    for i, l in enumerate(open(path, encoding='utf-8'), 1):
        l = l.strip()
        if not l: continue
        try: out.append(json.loads(l))
        except Exception as e: print('skipping malformed line', i, e)
    return out

print('results log:', RESULTS)

In [ ]:
# Audit of the 100M run (earlier audit showed it only ever logged step 0).
rows = read_jsonl(LOG100)
if rows:
    steps = [r.get('step', 0) for r in rows]
    print('rows:', len(rows), '| max step:', max(steps), '| last student_ppl:', rows[-1].get('student_ppl'))
    if max(steps) == 0: print('WARNING: the 100M run never advanced past step 0. Do not cite it as a result.')
else:
    print('100M log not visible from this account/folder. The earlier audit found only step 0 (student PPL ~1.9e20).')

## 3. Free analysis from the recovered records (no GPU)
The executed `DriveFolders` notebook showed these rows carry `metrics.wiki_ppl`, `metrics.lambada_acc`,
`metrics.long_nll` and `metrics.long_gap` for the 4M-token students. From those outputs:

| Student (4M tokens, 3 seeds) | WikiText PPL (approx. mean) | LAMBADA acc (approx. mean) |
|---|---|---|
| FP student | 27.3 | 41.9% |
| Ternary TWN | 75.5 | 18.9% |
| Ternary abs-mean | 115.4 | 13.2% |

So ternary costs far more on downstream tasks than the ~38% FineWeb-PPL gap suggests. The cell below recomputes
this from the file. **Check whether the `v2_best` rows have these metrics** (they were not visible in the printed head).
`long_gap` looks like a per-length-bucket student-minus-teacher NLL; confirm that in your evaluator code before citing it.

In [ ]:
rec = read_jsonl(RECOVERED)
print('recovered records:', len(rec))
rdf = pd.json_normalize(rec) if rec else pd.DataFrame()

def pick_ppl_col(df):
    for c in ['metrics.ppl', 'ppl', 'final_ppl', 'metrics.fineweb_ppl', 'metrics.final_ppl']:
        if c in df.columns: return c
    cands = [c for c in df.columns if 'ppl' in c.lower() and c not in ('init_ppl', 'metrics.wiki_ppl')]
    return cands[0] if cands else None

def mean_list(x): return float(np.mean(x)) if isinstance(x, (list, tuple)) and len(x) else np.nan
def growth(x):    return float(x[-1] - x[0]) if isinstance(x, (list, tuple)) and len(x) > 1 else np.nan

if len(rdf):
    PPL = pick_ppl_col(rdf); print('FineWeb PPL column:', PPL)
    if 'metrics.long_gap' in rdf: 
        rdf['long_gap_mean']   = rdf['metrics.long_gap'].map(mean_list)
        rdf['long_gap_growth'] = rdf['metrics.long_gap'].map(growth)
    cols = [c for c in [PPL, 'metrics.wiki_ppl', 'metrics.lambada_acc', 'long_gap_mean', 'long_gap_growth', 'train_s']
            if c and c in rdf.columns]
    summary = rdf.groupby('name')[cols].agg(['count', 'mean']).round(4) if 'name' in rdf else None
    display(summary)
    v2rows = rdf[rdf['name'].astype(str).str.contains('v2')] if 'name' in rdf else pd.DataFrame()
    print('v2_* rows:', len(v2rows))
    if len(v2rows):
        has = {c: int(v2rows[c].notna().sum()) for c in ['metrics.wiki_ppl', 'metrics.lambada_acc', 'metrics.long_gap'] if c in v2rows}
        print('v2_* rows with downstream metrics:', has, '(0 means Section 9 must run them)')
else:
    print('No recovered records loaded. Upload shift_scan_runs_recovered*.jsonl or fix the mount.')

In [ ]:
# Runtime planning from measured training times (T4). Uses the recovered train_s when available.
SEEDS = [0, 1, 2]
t = (rdf.groupby('name')['train_s'].mean() / 60) if len(rdf) and 'train_s' in rdf and 'name' in rdf else pd.Series(dtype=float)
fp_min   = float(t.get('fp_student', 36.8))     # measured ~36.8 min at 4M tokens
tern_min = float(t.get('tern_twn', 42.6))       # measured ~42.6 min at 4M tokens
n_fp_cells, n_tern_cells = 4, 4                 # A0 + 3x A2 (FP)  |  A1 + 3x A3 (ternary)
total_4m = (n_fp_cells * fp_min + n_tern_cells * tern_min) * len(SEEDS) / 60
print(f'FP run ~{fp_min:.1f} min, ternary run ~{tern_min:.1f} min at 4M tokens')
print(f'Full 24-run matrix: ~{total_4m:.1f} h at 4M tokens, ~{total_4m/2:.1f} h at 2M tokens (single T4)')
print('Colab sessions end, so the runner below is resumable. Run PRIORITY order first.')

## 4. Wire your real training / eval code (the one manual step)
Neither Drive view contains the trainer as a `.py` file. It lives inside `Shift_Scan_v2_Final_Colab.ipynb`.
The helper lists the functions defined in each code cell so you can pick the right cells, then runs them in this session.
Review the cells before executing them.

In [ ]:
def list_notebook_defs(nb_path):
    nb = json.load(open(nb_path))
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] != 'code': continue
        src = ''.join(c['source'])
        names = [l.split('(')[0].replace('def ', '').replace('class ', '').strip()
                 for l in src.splitlines() if l.startswith('def ') or l.startswith('class ')]
        if names: print(f'cell {i:3d}:', names[:8])
    return nb

def exec_cells(nb, idx):
    # Runs the chosen cells of the v2 notebook in this namespace. Review them first.
    for i in idx:
        src = ''.join(nb['cells'][i]['source'])
        src = '\n'.join(l for l in src.splitlines() if not l.lstrip().startswith(('!', '%')))
        exec(compile(src, f'v2_cell_{i}', 'exec'), globals())

if V2_NB:
    V2_NBJ = list_notebook_defs(V2_NB)
else:
    print('Shift_Scan_v2_Final_Colab.ipynb not visible from this account. Open it from the other account or copy it into this Drive.')

In [ ]:
TOKENS_MAIN, SEQ_MAIN = 4_000_000, 1024      # comparable to 31.83 PPL (set SEQ_MAIN to what v2 used)
TOKENS_PILOT, SEQ_PILOT = 2_000_000, 512     # the pilot protocol in your DriveFolders scaffold
PROTOCOL = 'pilot'                            # 'pilot' first, 'main' after the pilot is clean
TOKENS, SEQ_LEN = (TOKENS_PILOT, SEQ_PILOT) if PROTOCOL == 'pilot' else (TOKENS_MAIN, SEQ_MAIN)
PROTOCOL_ID = f'matched_{TOKENS // 1_000_000}M_seq{SEQ_LEN}_seeds012_v1'

def train_and_eval(cfg: dict, seed: int) -> dict:
    # Must: set seed; build the model; train for cfg['tokens']; evaluate on the SAME held-out FineWeb-Edu set as v2;
    # return dict(ppl=..., ce=..., tokens_actual=..., runtime_s=..., peak_vram_gb=..., ckpt_path=...).
    raise NotImplementedError('Wire this to the v2 train/eval functions (see the cell above).')

print('protocol:', PROTOCOL_ID)

## 5. Matched factorial: projections x decay (the main missing experiment)
Same data stream, validation set, tokenizer, init, optimizer, schedule, token budget and seq length in every cell.
Every row is stamped with `protocol_id` so pilot (2M) and main (4M) results are never mixed.

In [ ]:
base_q = dict(BEST['qcfg'])

def make_cfg(ternary, decay):
    q = dict(base_q)
    q['ternary'] = ternary
    q['decay'] = decay
    if decay == 'harpq':   q.update(decay_quantizer='horizon', decay_horizon=128)
    if decay == 'dyadic2': q.update(dyadic_terms=2)
    if ternary is None:    q['fp_proj'] = []      # nothing to protect when projections are FP
    return dict(qcfg=q, lr=BEST['lr'], tokens=TOKENS, seq_len=SEQ_LEN, model_size='130m')

CELLS = {
    'A0_fp_origdecay'   : make_cfg(None,  None),
    'A1_tern_origdecay' : make_cfg('twn', None),
    'A2_fp_nearest'     : make_cfg(None,  'nearest'),
    'A2_fp_harpq'       : make_cfg(None,  'harpq'),
    'A2_fp_dyadic2'     : make_cfg(None,  'dyadic2'),
    'A3_tern_nearest'   : make_cfg('twn', 'nearest'),
    'A3_tern_harpq'     : make_cfg('twn', 'harpq'),
    'A3_tern_dyadic2'   : make_cfg('twn', 'dyadic2'),
}
# Most informative first: does PoT decay hurt FP? does it hurt ternary?
PRIORITY = ['A0_fp_origdecay', 'A1_tern_origdecay', 'A3_tern_harpq', 'A3_tern_nearest',
            'A2_fp_harpq', 'A2_fp_nearest', 'A3_tern_dyadic2', 'A2_fp_dyadic2']

RUN = False   # flip to True only after train_and_eval is wired
done = {(r['protocol_id'], r['cell'], r['seed']) for r in read_jsonl(RESULTS) if r.get('status') == 'ok' and 'protocol_id' in r}

if RUN:
    for name in PRIORITY:
        for seed in SEEDS:
            if (PROTOCOL_ID, name, seed) in done: continue
            t0 = time.time()
            try:
                m = train_and_eval(CELLS[name], seed)
                row = dict(protocol_id=PROTOCOL_ID, cell=name, seed=seed, tokens_budget=TOKENS, seq_len=SEQ_LEN,
                           status='ok', runtime_s=time.time() - t0, **m)
            except Exception as e:
                row = dict(protocol_id=PROTOCOL_ID, cell=name, seed=seed, status='error', error=repr(e))
            append_jsonl(RESULTS, row)
            print(name, seed, row.get('ppl'), row['status'])
else:
    print('RUN=False. Planned runs:', len(CELLS) * len(SEEDS), '| already done:', len(done))

## 6. Aggregate, significance, and a power check
Your v2-best seed std is ~0.096 PPL. The check below converts that into the smallest difference 3 seeds can reliably detect.
Observed decay-method differences so far (~0.03 PPL) are far below it, so they cannot support a ranking.

In [ ]:
from scipy import stats

sd_ref = 0.09587    # seed std of v2_best (3 seeds, 4M tokens)
def mde(sd, n, alpha=0.05, power=0.8):
    dfree = 2 * n - 2
    return (stats.t.ppf(1 - alpha / 2, dfree) + stats.t.ppf(power, dfree)) * sd * math.sqrt(2 / n)

for n in (3, 5, 8):
    print(f'seeds={n}: smallest reliably detectable PPL difference ~ {mde(sd_ref, n):.3f}')
print('Observed Nearest-vs-v2_best gap in recovered data: ~0.026 PPL (1 seed each).')

In [ ]:
df = pd.DataFrame([r for r in read_jsonl(RESULTS) if r.get('status') == 'ok'])
if len(df):
    for pid, g in df.groupby('protocol_id'):
        print('\nProtocol:', pid)
        display(g.groupby('cell')['ppl'].agg(['count', 'mean', 'std']).round(4))

        def compare(a, b):
            x = g[g.cell == a].sort_values('seed').ppl.values
            y = g[g.cell == b].sort_values('seed').ppl.values
            if len(x) < 2 or len(y) < 2: return None
            t = stats.ttest_ind(x, y, equal_var=False)
            return dict(a=a, b=b, diff=float(y.mean() - x.mean()), p=float(t.pvalue))

        pairs = [('A1_tern_origdecay', 'A3_tern_harpq'), ('A1_tern_origdecay', 'A3_tern_nearest'),
                 ('A1_tern_origdecay', 'A3_tern_dyadic2'), ('A3_tern_nearest', 'A3_tern_harpq'),
                 ('A0_fp_origdecay', 'A2_fp_harpq'), ('A0_fp_origdecay', 'A2_fp_nearest'),
                 ('A0_fp_origdecay', 'A1_tern_origdecay')]
        display(pd.DataFrame([c for c in (compare(a, b) for a, b in pairs) if c]))
    print('Claim a decay effect only if the gap clears the power threshold above AND p < 0.05.')
else:
    print('No finished runs yet.')

## 7. Component sensitivity beyond projections
Per-layer projection sensitivity is already done (`stage1_sensitivity.csv`). Still missing: **dt, B, C, decay, state**.
The hooks fake-quantize slices of `in_proj`'s output at N bits. For Mamba-2 the output is `[z, x, B, C, dt]`; verify the
split sizes against your model class before trusting results.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, torch.cuda.get_device_name(0) if dev == 'cuda' else '')

def fake_quant(x, bits):
    qmax = 2 ** (bits - 1) - 1
    s = x.detach().abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / qmax
    return (x / s).round().clamp(-qmax, qmax) * s

def split_sizes(mixer):
    d_inner = getattr(mixer, 'd_inner', None) or mixer.intermediate_size
    d_state = getattr(mixer, 'd_state', None) or mixer.ssm_state_size
    ngroups = getattr(mixer, 'ngroups', None) or getattr(mixer, 'n_groups', 1)
    nheads  = getattr(mixer, 'nheads', None) or mixer.num_heads
    return dict(z=d_inner, x=d_inner, B=ngroups * d_state, C=ngroups * d_state, dt=nheads)

def attach_slice_quant(model, part, bits):
    handles = []
    for m in model.modules():
        if hasattr(m, 'in_proj') and (hasattr(m, 'd_state') or hasattr(m, 'ssm_state_size')):
            sz = split_sizes(m); names = list(sz); idx = np.cumsum([0] + [sz[n] for n in names])
            lo, hi = int(idx[names.index(part)]), int(idx[names.index(part) + 1])
            def hook(mod, inp, out, lo=lo, hi=hi):
                out = out.clone(); out[..., lo:hi] = fake_quant(out[..., lo:hi], bits); return out
            handles.append(m.in_proj.register_forward_hook(hook))
    return handles

# Usage once a model and validation blocks exist (Section 8 builds them):
# for part in ['dt', 'B', 'C']:
#     for bits in [8, 6, 4]:
#         hs = attach_slice_quant(model, part, bits)
#         ppl = eval_ppl(model, val_blocks)
#         append_jsonl(RESULTS, dict(protocol_id='sens_v1', cell=f'sens_{part}_{bits}', seed=0, status='ok', ppl=ppl))
#         [h.remove() for h in hs]
# Decay and recurrent state: reuse your nearest/harpq/dyadic2 quantizers and a rounding hook inside the scan.

## 8. Long-context validation on the existing 4M checkpoint
No retraining needed. Compare FP student, ternary (original decay) and any A3 variant at lengths beyond training length,
using one fixed validation stream. Report loss by position and count non-finite logits. The recovered `long_gap` data in
Section 3 is the first thing to compare against.

In [ ]:
@torch.no_grad()
def eval_ppl(model, blocks, bs=4):
    model.eval(); tot, n = 0.0, 0
    for i in range(0, len(blocks), bs):
        x = torch.stack(blocks[i:i + bs]).to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            logits = model(x).logits
        loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction='sum')
        tot += loss.item(); n += x[:, 1:].numel()
    return math.exp(tot / n)

@torch.no_grad()
def loss_by_position(model, blocks, bucket=128, bs=1):
    model.eval(); per_pos = None; count = 0; nonfinite = 0
    for i in range(0, len(blocks), bs):
        x = torch.stack(blocks[i:i + bs]).to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            logits = model(x).logits
        nonfinite += int((~torch.isfinite(logits)).sum().item())
        l = F.cross_entropy(logits[:, :-1].float().transpose(1, 2), x[:, 1:], reduction='none').mean(0)
        per_pos = l if per_pos is None else per_pos + l; count += 1
    per_pos = (per_pos / count).cpu().numpy()
    nb = len(per_pos) // bucket
    return dict(bucket_loss=[float(per_pos[j * bucket:(j + 1) * bucket].mean()) for j in range(nb)], nonfinite=nonfinite)

def build_val_blocks(tokenizer, seq_len, n_blocks, val_modulo=100, max_docs=20000):
    # IMPORTANT: replace with your v2 validation builder so numbers stay comparable with 31.83.
    from datasets import load_dataset
    ds = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT', split='train', streaming=True)
    buf, blocks = [], []
    for i, ex in enumerate(ds):
        if i >= max_docs: break
        if i % val_modulo != 0: continue
        buf += tokenizer(ex['text'])['input_ids'] + [tokenizer.eos_token_id]
        while len(buf) >= seq_len and len(blocks) < n_blocks:
            blocks.append(torch.tensor(buf[:seq_len])); buf = buf[seq_len:]
        if len(blocks) >= n_blocks: break
    return blocks

# Inspect the checkpoint first (your own file; weights_only=False runs pickle, so only load files you trust).
if CKPT and Path(CKPT).exists():
    ck = torch.load(CKPT, map_location='cpu', weights_only=False)
    print(type(ck)); 
    if isinstance(ck, dict): print(list(ck.keys())[:20])
else:
    print('Checkpoint not found. Unzip the v2 archive (Section 2) or run in the account that holds the folder.')

LENGTHS = [1024, 2048, 4096, 8192]
# for L in LENGTHS:
#     blocks = build_val_blocks(tok, L, n_blocks=16)
#     for name, model in {'A0': m_fp, 'A1': m_tern, 'A3': m_tern_pot}.items():
#         r = loss_by_position(model, blocks)
#         append_jsonl(RESULTS, dict(protocol_id='longctx_v1', cell='longctx_' + name, seed=0, seq_len=L, status='ok', **r))

## 9. Zero-shot benchmarks for the rows that lack them
The recovered rows already hold WikiText and LAMBADA for FP / abs-mean / TWN students at 4M tokens. Run this only for
what is missing: the teacher, `v2_best` (31.8 PPL), and the A-cells. Add HellaSwag, ARC-easy and PIQA if time allows.
Needs `lm_eval` (0.4.13 is in your run manifest).

In [ ]:
# !pip -q install lm_eval==0.4.13
TASKS = ['lambada_openai', 'wikitext', 'hellaswag', 'arc_easy', 'piqa']

def run_zero_shot(model, tokenizer, tag):
    import lm_eval
    from lm_eval.models.huggingface import HFLM
    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=8)
    res = lm_eval.simple_evaluate(model=lm, tasks=TASKS, num_fewshot=0)['results']
    append_jsonl(RESULTS, dict(protocol_id='zeroshot_v1', cell='zeroshot_' + tag, seed=0, status='ok', results=res))
    return res

## 10. Real hardware evaluation
The Triton kernel was ~82x slower than FP16 (12.98 ms vs 0.159 ms at M=512), so nothing here can claim a speedup yet.
A fairer first test is ternary weights stored as int8 with int8 activations (`torch._int_mm`, int8 tensor cores on a T4).
If that beats FP16 it is deployable today; a bit-packed shift/add kernel is a later optimization.

In [ ]:
def bench(fn, iters=50, warm=10):
    for _ in range(warm): fn()
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

def kernel_bench(M, K, N):
    out = {}
    x16 = torch.randn(M, K, device='cuda', dtype=torch.float16)
    w16 = torch.randn(K, N, device='cuda', dtype=torch.float16)
    out['fp16_ms'] = bench(lambda: x16 @ w16)
    wt = torch.randint(-1, 2, (K, N), device='cuda', dtype=torch.int8)
    x8 = torch.randint(-127, 128, (M, K), device='cuda', dtype=torch.int8)
    try:
        out['int8_ternary_ms'] = bench(lambda: torch._int_mm(x8, wt))
    except Exception as e:
        out['int8_ternary_ms'] = None; out['note'] = repr(e)[:120]
    return out

if torch.cuda.is_available():
    for (M, K, N) in [(512, 768, 3072), (2048, 768, 3072), (2048, 1536, 768)]:
        r = kernel_bench(M, K, N); r.update(M=M, K=K, N=N); print(r)
        append_jsonl(RESULTS, dict(protocol_id='kernel_v1', cell='kernel_bench', seed=0, status='ok', **r))
else:
    print('No GPU. Switch Runtime to T4 GPU for the kernel benchmark.')
# Also measure end-to-end tokens/sec and peak VRAM for the FP16 vs ternary model.

## 11. Claim check
Fill from `runs_last.jsonl` once runs finish. Claim only what survives.

In [ ]:
CLAIMS = {
 'Ternary TWN + FP-sensitive layers trains to ~31.8 PPL @ 4M (3 seeds)'          : 'supported',
 'Ternary keeps downstream ability'                                                : 'NOT supported: recovered LAMBADA ~19% (TWN) vs ~42% (FP); recheck on v2_best',
 'PoT decay preserves quality (FP projections)'                                   : 'needs A0 vs A2, 3 seeds',
 'PoT decay preserves quality (ternary projections)'                              : 'needs A1 vs A3, 3 seeds',
 'HARPQ beats Nearest PoT'                                                        : 'NOT supported (differences ~0.03 PPL, below the 3-seed detection limit)',
 'Long-context stability'                                                         : 'needs Section 8 (check recovered long_gap first)',
 'Multiplication elimination (85% SSM / 62% whole model)'                         : 'analytical only; state as such',
 'Inference speedup'                                                              : 'NOT supported (kernel ~82x slower)',
 'Novelty: ternary SSM'                                                           : 'NOT novel (prior work exists)',
 'Novelty: PoT decay + ternary + sensitivity + shift-add kernel, matched'         : 'only if A3 shows no loss vs A1',
}
display(pd.DataFrame(list(CLAIMS.items()), columns=['claim', 'status']))

## 12. Paper checklist
- [ ] Matched A0-A3 table: mean +/- std, 3 seeds, one protocol per table (pilot and main reported separately)
- [ ] Power statement: with 3 seeds only differences above ~0.3 PPL are detectable
- [ ] Per-layer sensitivity figure from `stage1_sensitivity.csv` (justifies keeping `0:out, 13:in, 23:out, 0:in, 3:in, 23:in` in FP)
- [ ] Training-recipe ablation table from `stage2_ablation.csv` (ramp and FP-sensitive are the big levers)
- [ ] Downstream table: WikiText / LAMBADA for teacher, FP student, TWN, `v2_best`
- [ ] Long-context loss-by-position plot and non-finite counts
- [ ] Kernel/latency table, with the negative result stated if the kernel is still slower
- [ ] Limitations: 130M scale, at most 12M tokens, T4 only, two-account data split, lost checkpoints, 100M run not completed
- [ ] Do not claim HARPQ superiority unless Section 5-6 support it